# ✈️ T10 Flight Option Finder — Master Pipeline Demonstration

This notebook demonstrates the **complete real pipeline** for the autonomous Flight Option Finder agent.

**Pipeline Architecture:**
1. Environment check & config
2. Natural language user request
3. Structured request extraction
4. Internal airport resolution (City → IATA)
5. AviationStack flight schedule query & capability verification
6. Flight schedule results
7. Ignav authoritative live fare query
8. Raw fare results
9. Deterministic matching engine
10. Matched fare validation & status preservation
11. Hard constraint & budget filtering
12. Candidate preference ranking & decision
13. Real booking link retrieval (Ignav booking-links API)
14. Auxiliary requests + BeautifulSoup web verification
15. Final recommendation presentation

## 1. Environment Check & Configuration
Validates presence of required API keys and masks credentials.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from config import config

print('Config Status:')
print(repr(config))
missing = config.validate(check_nvidia=False)
if missing:
    print(f'Warning: Missing keys: {missing}')
else:
    print('✓ Core configurations loaded and validated.')

Config Status:
Config(aviationstack_api_key=7b24...8398, ignav_api_key=igna...04o0, nvidia_api_key=nvap...u2i9, nvidia_base_url='https://integrate.api.nvidia.com/v1', nvidia_model='meta/llama-3.3-70b-instruct', ignav_base_url='https://ignav.com/api', aviationstack_base_url='http://api.aviationstack.com/v1')
✓ Core configurations loaded and validated.


## 2. User Request
Define the user's natural language flight search query.

In [2]:
travel_date = '2026-12-04'
user_query = f"Find me a flight from Delhi to Mumbai on {travel_date} under ₹8,000"
print(f'User Query: \"{user_query}\"')


User Query: "Find me a flight from Delhi to Mumbai on 2026-12-04 under ₹8,000"


## 3. Parsed Request
Extract structured entities (origin, destination, date, budget, currency, cabin class).

In [3]:
from agent import agent

search_req = agent.parse_user_query(user_query)
print('Extracted Search Parameters:')
for k, v in search_req.model_dump().items():
    print(f'  - {k}: {v}')

NVIDIA NIM request interpretation unavailable; using deterministic parsing: NVIDIA NIM model 'meta/llama-3.3-70b-instruct' is deprecated or gone (HTTP 410): {"type":"about:blank","title":"Gone","status":410,"detail":"The model 'meta/llama-3.3-70b-instruct' has reached its end of life on 2026-08-26T09:00:00Z and is no longer available."}



Extracted Search Parameters:
  - origin_city: Delhi
  - destination_city: Mumbai
  - origin_code: None
  - destination_code: None
  - travel_date: 2026-12-04
  - budget: 8000.0
  - currency: INR
  - passengers: 1
  - cabin_class: economy
  - recommendation_profile: BEST_BALANCED
  - preferred_airline: None
  - preferred_departure: None
  - max_stops: None


## 4. Airport Resolution
Resolve city names to standard 3-letter IATA codes internally.

In [4]:
from airport_resolver import resolve_airport

orig_res = resolve_airport(search_req.origin_city)
dest_res = resolve_airport(search_req.destination_city)

print(f'Origin: {search_req.origin_city} -> {orig_res.code} ({orig_res.airport_name})')
print(f'Destination: {search_req.destination_city} -> {dest_res.code} ({dest_res.airport_name})')

Origin: Delhi -> DEL (Indira Gandhi International Airport)
Destination: Mumbai -> BOM (Chhatrapati Shivaji Maharaj International Airport)


## 5. AviationStack Search & Capability Verification
Query AviationStack and verify account plan capability for requested date.

In [5]:
from tools import search_flights

flights_resp = search_flights(
    origin=orig_res.code,
    destination=dest_res.code,
    travel_date=search_req.travel_date
)
print(f'AviationStack Status: {flights_resp.get("status")}')
if flights_resp.get('message'):
    print(f'Note: {flights_resp.get("message")}')

AviationStack Status: future_date_unavailable
Note: AviationStack future-date capability unavailable for the configured plan/account: Access Restricted - Your current Subscription Plan does not support this API Function.


## 6. Flight Schedule Results
Inspect retrieved flight options.

In [6]:
flights = flights_resp.get('flights', [])
print(f'Retrieved {len(flights)} flight schedule records.')
for f in flights[:5]:
    print(f'  {f.get("airline")} {f.get("flight_number")} | {f.get("origin")}->{f.get("destination")} | {f.get("departure_time")}->{f.get("arrival_time")}')

Retrieved 0 flight schedule records.


## 7. Ignav Fare Search
Fetch real, authoritative fares and itinerary segments from Ignav.

In [7]:
from ignav_client import ignav_client

raw_fares = ignav_client.search_one_way_fares(
    origin=orig_res.code,
    destination=dest_res.code,
    departure_date=search_req.travel_date,
    adults=search_req.passengers,
    cabin_class=search_req.cabin_class,
)
print(f'Retrieved {len(raw_fares)} live fares from Ignav.')

Retrieved 57 live fares from Ignav.


## 8. Fare Results
Display raw fares retrieved from Ignav.

In [8]:
for fare in raw_fares[:5]:
    print(f'  {fare.airline} {fare.flight_number} | {fare.origin}->{fare.destination} | {fare.departure_time}->{fare.arrival_time} | ₹{fare.price:,.0f} [{fare.price_status.upper()}] | ID: {fare.ignav_id[:16]}...')

  IndiGo 6E353 | DEL->BOM | 03:30->05:55 | ₹6,551 [VERIFIED] | ID: d08493c535984246...
  IndiGo 6E449 | DEL->BOM | 05:00->07:25 | ₹6,551 [VERIFIED] | ID: 6cf49494398e4051...
  IndiGo 6E6676 | DEL->BOM | 12:30->14:55 | ₹6,551 [VERIFIED] | ID: a2b20d82a4e74daf...
  IndiGo 6E864 | DEL->BOM | 15:30->17:55 | ₹6,551 [VERIFIED] | ID: 211b3152d97e4661...
  IndiGo 6E324 | DEL->BOM | 13:15->15:45 | ₹6,551 [VERIFIED] | ID: ad1366c9a4964e39...


## 9. Matching Engine
Execute deterministic matching rules (AviationStack flight schedules ⟷ Ignav fares).

In [9]:
from matching import match_flights_with_fares
from models import FlightOption

# Run matching engine
flight_objs = [FlightOption(**f) for f in flights] if flights else []
if flight_objs:
    matched_fares = match_flights_with_fares(flight_objs, raw_fares)
else:
    # Fallback to direct Ignav itineraries when flight schedules are restricted on free tier
    matched_fares = raw_fares

print(f'Matched options total: {len(matched_fares)}')

Matched options total: 57


## 10. Verified Fare Results
Inspect matched fare outcomes and status preservation.

In [10]:
verified_count = sum(1 for f in matched_fares if f.price_status == 'verified')
unverified_count = sum(1 for f in matched_fares if f.price_status == 'unverified')
print(f'Verified Fares: {verified_count} | Unverified Fares: {unverified_count}')

Verified Fares: 57 | Unverified Fares: 0


## 11. Budget Filtering
Enforce deterministic hard constraint: price <= budget.

In [11]:
budget = search_req.budget or 8000.0
valid_candidates = [f for f in matched_fares if f.price_available and f.price is not None and f.price <= budget]
print(f'Valid options within budget (≤ ₹{budget:,.0f}): {len(valid_candidates)}')
for c in valid_candidates[:5]:
    print(f'  ✓ {c.airline} {c.flight_number}: ₹{c.price:,.0f} ({c.departure_time} - {c.arrival_time})')

Valid options within budget (≤ ₹8,000): 54
  ✓ IndiGo 6E353: ₹6,551 (03:30 - 05:55)
  ✓ IndiGo 6E449: ₹6,551 (05:00 - 07:25)
  ✓ IndiGo 6E6676: ₹6,551 (12:30 - 14:55)
  ✓ IndiGo 6E864: ₹6,551 (15:30 - 17:55)
  ✓ IndiGo 6E324: ₹6,551 (13:15 - 15:45)


## 12. Decision & Ranking
Rank valid candidates by verification status, preferences, price, and duration.

In [12]:
best_flight = None
if valid_candidates:
    valid_candidates.sort(key=lambda x: (0 if x.price_status == 'verified' else 1, x.price if x.price else float('inf')))
    best_flight = valid_candidates[0]
    print(f'Top Selected Flight: {best_flight.airline} {best_flight.flight_number} at ₹{best_flight.price:,.0f} [{best_flight.price_status.upper()}]')
else:
    print('No candidates satisfied all criteria.')

Top Selected Flight: IndiGo 6E353 at ₹6,551 [VERIFIED]


## 13. Booking Link Retrieval
Fetch real provider booking link from Ignav via `POST /api/fares/booking-links` using `ignav_id`.

In [13]:
booking_opt = None
if best_flight and best_flight.ignav_id:
    booking_opts = ignav_client.get_booking_links(best_flight.ignav_id)
    print(f'Retrieved {len(booking_opts)} booking options:')
    for opt in booking_opts:
        print(f'  - Provider: {opt.provider} ({opt.provider_type}) | URL: {opt.url[:70]}...')
    if booking_opts:
        booking_opt = booking_opts[0]
        best_flight.booking_url = booking_opt.url
        best_flight.booking_provider = booking_opt.provider

Retrieved 1 booking options:
  - Provider: IndiGo (airline) | URL: https://www.goindigo.in/flight-select-redirection.html?cid=metasearch|...


## 14. Requests + BeautifulSoup Web Verification
Auxiliary HTTPS inspection of the live booking page using requests + BeautifulSoup.

In [14]:
from scraping.booking_verifier import booking_verifier

if best_flight and best_flight.booking_url:
    ver_result = booking_verifier.verify_booking_page(
        booking_url=best_flight.booking_url,
        expected_flight=best_flight,
        expected_fare=best_flight.price
    )
    print(f'Web Verification Status: {ver_result.status.upper()}')
    print(f'Details: {ver_result.details}')
    best_flight.web_verification_status = ver_result.status
else:
    print('No booking URL available for web verification.')

Web Verification Status: UNAVAILABLE
Details: Timeout connecting to booking provider page.


## 15. Final Recommendation Presentation
Reuse the earlier route and fare results and print the selected recommendation without re-running provider calls.

In [15]:
best_flight = None
if valid_candidates:
    best_flight = sorted(valid_candidates, key=lambda fare: (fare.price is None, fare.price if fare.price is not None else float('inf')))[0]
if best_flight:
    print(f'Notebook reused {len(flights)} schedule records and {len(raw_fares)} fare quotes; selected {best_flight.airline} {best_flight.flight_number} at ₹{best_flight.price:,.0f}.')
else:
    print(f'Notebook reused {len(flights)} schedule records and {len(raw_fares)} fare quotes; no eligible candidate within budget.')


Notebook reused 0 schedule records and 57 fare quotes; selected IndiGo 6E353 at ₹6,551.
